# Sink Node Characterisation — Which Nodes Become Sinks?

**Hypothesis H2:** Attention sinks in graph transformers preferentially form on nodes with specific structural properties (high degree, high centrality, spectral extremity).

This notebook re-runs inference on the no-VNode baseline to extract **per-node** sink scores and correlates them with graph-theoretic node properties.

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import yaml
import pickle

from src.model import InstrumentedGPS
from src.datasets import get_dataloaders, DATASET_INFO, compute_spectral_properties
from src.metrics import compute_sink_scores

ModuleNotFoundError: No module named 'torch_geometric'

In [ ]:
# Load config
experiment_id = 'zinc-novnode-rwse-10L'
config_path = f'../outputs/{experiment_id}/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

# Load dataset info and data
dataset_info = DATASET_INFO[config['data']['dataset']].copy()
_, _, test_loader, _ = get_dataloaders(config)

# Build model and load weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = InstrumentedGPS(config, dataset_info).to(device)
model.load_state_dict(torch.load(f'../outputs/{experiment_id}/best_model.pt', map_location=device, weights_only=True))
model.eval()
print(f"Loaded {experiment_id} on {device}")

## 1. Extract per-node sink scores and node properties

In [ ]:
from torch_geometric.utils import degree
from tqdm import tqdm

model._register_attn_hooks()

# For each graph, collect: per-node sink scores at each layer, node degree, num_nodes
all_graphs = []
max_graphs = 200

graphs_done = 0
for batch in tqdm(test_loader, desc='Extracting sink data'):
    if graphs_done >= max_graphs:
        break
    batch = batch.to(device)
    with torch.no_grad():
        _ = model(batch, collect_diagnostics=True)
    
    batch_ids = model.layer_data[0]['batch']
    unique_graphs = batch_ids.unique()
    
    for g_idx, g_id in enumerate(unique_graphs):
        if graphs_done >= max_graphs:
            break
        
        graph_mask = (batch_ids == g_id)
        num_nodes_g = graph_mask.sum().item()
        
        # Node degree from edge_index
        g_edges = batch.edge_index[:, (batch.batch[batch.edge_index[0]] == g_id)]
        # Remap to local indices
        node_ids = torch.where(graph_mask)[0]
        local_map = {int(n): i for i, n in enumerate(node_ids)}
        local_src = torch.tensor([local_map[int(s)] for s in g_edges[0] if int(s) in local_map])
        deg = degree(local_src, num_nodes=num_nodes_g)
        
        # Spectral properties
        # Get Fiedler vector (eigenvector of lambda_2)
        from torch_geometric.utils import get_laplacian, to_scipy_sparse_matrix
        # Build local edge_index
        local_edges = []
        for s, d in g_edges.t().tolist():
            if s in local_map and d in local_map:
                local_edges.append([local_map[s], local_map[d]])
        if local_edges:
            local_ei = torch.tensor(local_edges).t()
        else:
            local_ei = torch.zeros(2, 0, dtype=torch.long)
        
        ei_lap, ew_lap = get_laplacian(local_ei, normalization='sym', num_nodes=num_nodes_g)
        L = to_scipy_sparse_matrix(ei_lap, ew_lap, num_nodes=num_nodes_g).toarray()
        eigenvalues, eigenvectors = np.linalg.eigh(L)
        # Fiedler vector = eigenvector of 2nd smallest eigenvalue
        fiedler = eigenvectors[:, 1] if num_nodes_g > 1 else np.zeros(num_nodes_g)
        
        # Betweenness centrality (approximate via networkx for small graphs)
        import networkx as nx
        G = nx.Graph()
        G.add_nodes_from(range(num_nodes_g))
        for s, d in zip(local_ei[0].tolist(), local_ei[1].tolist()):
            G.add_edge(s, d)
        centrality = nx.betweenness_centrality(G)
        cent_array = np.array([centrality.get(i, 0.0) for i in range(num_nodes_g)])
        
        # Per-node sink scores at each layer (use last few layers where sinks are strongest)
        graph_data = {
            'num_nodes': num_nodes_g,
            'degree': deg.numpy(),
            'fiedler': fiedler,
            'centrality': cent_array,
            'sink_scores_by_layer': {},
        }
        
        for layer_idx in range(1, model.num_layers + 1):
            attn_full = model.attn_weights[layer_idx - 1]
            if attn_full is not None and g_idx < attn_full.size(0):
                attn_g = attn_full[g_idx, :, :num_nodes_g, :num_nodes_g]
                sink_data = compute_sink_scores(attn_g)
                graph_data['sink_scores_by_layer'][layer_idx] = sink_data['sink_score_per_node'].numpy()
        
        all_graphs.append(graph_data)
        graphs_done += 1

model._remove_attn_hooks()
print(f"Collected data for {len(all_graphs)} graphs")

## 2. Sink score vs Node Degree

In [ ]:
# Collect (degree, sink_score) pairs from last layer
last_layer = model.num_layers
degrees = []
sink_scores = []

for g in all_graphs:
    if last_layer in g['sink_scores_by_layer']:
        scores = g['sink_scores_by_layer'][last_layer]
        degrees.extend(g['degree'].tolist())
        sink_scores.extend(scores.tolist())

degrees = np.array(degrees)
sink_scores = np.array(sink_scores)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: Sink score vs degree
axes[0].scatter(degrees, sink_scores, alpha=0.15, s=8, color='tab:red')
r_deg, p_deg = spearmanr(degrees, sink_scores)
axes[0].set_xlabel('Node Degree')
axes[0].set_ylabel('Sink Score (last layer)')
axes[0].set_title(f'Sink Score vs Degree (r={r_deg:.3f}, p={p_deg:.2e})')
axes[0].grid(True, alpha=0.3)

# Panel 2: Sink score vs |Fiedler value| (spectral extremity)
fiedler_vals = []
sink_scores_f = []
for g in all_graphs:
    if last_layer in g['sink_scores_by_layer']:
        scores = g['sink_scores_by_layer'][last_layer]
        fiedler_vals.extend(np.abs(g['fiedler']).tolist())
        sink_scores_f.extend(scores.tolist())

fiedler_vals = np.array(fiedler_vals)
sink_scores_f = np.array(sink_scores_f)

axes[1].scatter(fiedler_vals, sink_scores_f, alpha=0.15, s=8, color='tab:blue')
r_fied, p_fied = spearmanr(fiedler_vals, sink_scores_f)
axes[1].set_xlabel('|Fiedler Vector Value|')
axes[1].set_ylabel('Sink Score (last layer)')
axes[1].set_title(f'Sink Score vs Spectral Extremity (r={r_fied:.3f}, p={p_fied:.2e})')
axes[1].grid(True, alpha=0.3)

# Panel 3: Sink score vs betweenness centrality
cent_vals = []
sink_scores_c = []
for g in all_graphs:
    if last_layer in g['sink_scores_by_layer']:
        scores = g['sink_scores_by_layer'][last_layer]
        cent_vals.extend(g['centrality'].tolist())
        sink_scores_c.extend(scores.tolist())

cent_vals = np.array(cent_vals)
sink_scores_c = np.array(sink_scores_c)

axes[2].scatter(cent_vals, sink_scores_c, alpha=0.15, s=8, color='tab:green')
r_cent, p_cent = spearmanr(cent_vals, sink_scores_c)
axes[2].set_xlabel('Betweenness Centrality')
axes[2].set_ylabel('Sink Score (last layer)')
axes[2].set_title(f'Sink Score vs Centrality (r={r_cent:.3f}, p={p_cent:.2e})')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Figure 2: What Determines Sink Nodes?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'../outputs/figure2_sink_characterisation.pdf', bbox_inches='tight', dpi=150)
plt.show()

print(f"\nCorrelation summary:")
print(f"  Degree vs sink score:     r={r_deg:.3f}, p={p_deg:.2e}")
print(f"  |Fiedler| vs sink score:  r={r_fied:.3f}, p={p_fied:.2e}")
print(f"  Centrality vs sink score: r={r_cent:.3f}, p={p_cent:.2e}")

## 3. Sink node properties table

In [ ]:
# For each graph, identify the sink node and report its properties
print(f"{'Graph':>6} {'Nodes':>6} {'SinkNode':>8} {'SinkScore':>10} {'Degree':>7} {'DegRank':>8} {'|Fiedler|':>10} {'FiedRank':>9} {'Centrality':>11} {'CentRank':>9}")
print("-" * 105)

sink_is_max_degree = 0
sink_is_top3_degree = 0
total_valid = 0

for i, g in enumerate(all_graphs[:50]):  # Show first 50
    if last_layer not in g['sink_scores_by_layer']:
        continue
    
    scores = g['sink_scores_by_layer'][last_layer]
    sink_node = np.argmax(scores)
    sink_score = scores[sink_node]
    
    deg = g['degree']
    fied = np.abs(g['fiedler'])
    cent = g['centrality']
    
    # Ranks (1 = highest)
    deg_rank = int((deg >= deg[sink_node]).sum())
    fied_rank = int((fied >= fied[sink_node]).sum())
    cent_rank = int((cent >= cent[sink_node]).sum())
    
    n = g['num_nodes']
    total_valid += 1
    if deg_rank == 1:
        sink_is_max_degree += 1
    if deg_rank <= 3:
        sink_is_top3_degree += 1
    
    if i < 20:  # Print first 20
        print(f"{i:>6} {n:>6} {sink_node:>8} {sink_score:>10.4f} {deg[sink_node]:>7.0f} {deg_rank:>8}/{n} {fied[sink_node]:>10.4f} {fied_rank:>9}/{n} {cent[sink_node]:>11.4f} {cent_rank:>9}/{n}")

print(f"\nSummary over {total_valid} graphs:")
print(f"  Sink node is max-degree node: {sink_is_max_degree}/{total_valid} ({100*sink_is_max_degree/total_valid:.1f}%)")
print(f"  Sink node is top-3 degree:    {sink_is_top3_degree}/{total_valid} ({100*sink_is_top3_degree/total_valid:.1f}%)")

## 4. How stable are sinks across layers?

In [ ]:
# Track sink node identity across layers
fig, ax = plt.subplots(figsize=(10, 5))

consistency_scores = []
for g in all_graphs:
    sink_nodes_by_layer = []
    for l in sorted(g['sink_scores_by_layer'].keys()):
        scores = g['sink_scores_by_layer'][l]
        sink_nodes_by_layer.append(np.argmax(scores))
    
    if len(sink_nodes_by_layer) > 1:
        # Fraction of layers where the sink is the same as the final layer's sink
        final_sink = sink_nodes_by_layer[-1]
        consistency = np.mean([1 if s == final_sink else 0 for s in sink_nodes_by_layer])
        consistency_scores.append(consistency)

ax.hist(consistency_scores, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Consistency (fraction of layers with same sink as final layer)')
ax.set_ylabel('Number of graphs')
ax.set_title('Sink Node Consistency Across Layers')
ax.axvline(np.mean(consistency_scores), color='red', linestyle='--', 
           label=f'Mean: {np.mean(consistency_scores):.2f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'../outputs/figure2b_sink_consistency.pdf', bbox_inches='tight', dpi=150)
plt.show()

print(f"Mean consistency: {np.mean(consistency_scores):.3f}")
print(f"Std consistency: {np.std(consistency_scores):.3f}")